# ncORF Tool Consensus Analysis

## Goals
1. **Tool Calling Behavior**: What does each tool call? (feature counts, size distributions)
2. **Agreement Patterns**: Where do tools agree vs disagree?
3. **Genomic Context**: How does agreement vary by ORF classification (CDS, UTR, intergenic)?
4. **Boundary Analysis**: Do tools agree on starts/stops? Is there overlapping signal?

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict, Counter
import matplotlib.pyplot as plt
import seaborn as sns

# Set style
sns.set_style("whitegrid")
sns.set_palette("Set2")
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Configuration
RESULTS_DIR = "/hps/nobackup/flicek/ensembl/genebuild/jackt/riboseq/pilot/consensus_output/consensus_reports"
FOCUS_SAMPLE = None  # Set to specific sample name or None for all

## 1. Load Data

In [ ]:
def load_consensus_files(results_dir, focus_sample=None):
    results_dir = Path(results_dir)
    consensus_data = {}
    
    for tsv_file in results_dir.rglob('*.consensus.tsv'):
        parts = tsv_file.stem.split('.')
        if len(parts) >= 3:
            sample_name = parts[0]
            query_tool = parts[1]
            
            if focus_sample and sample_name != focus_sample:
                continue
            
            try:
                df = pd.read_csv(tsv_file, sep='\t')
                consensus_data[(sample_name, query_tool)] = df
                print(f"Loaded {sample_name} / {query_tool}: {len(df):,} features")
            except Exception as e:
                print(f"Warning: Failed to load {tsv_file}: {e}")
    
    return consensus_data

consensus_data = load_consensus_files(RESULTS_DIR, FOCUS_SAMPLE)
print(f"\nLoaded {len(consensus_data)} datasets")

## 2. Tool Calling Statistics

**Question**: What does each tool call? Are they detecting similar numbers of features?

In [ ]:
# Aggregate tool statistics across samples
tool_stats = []

for (sample, query_tool), df in consensus_data.items():
    lengths = (df['end_pos'] - df['start_pos']).values
    
    tool_stats.append({
        'sample': sample,
        'tool': query_tool,
        'num_features': len(df),
        'mean_length': lengths.mean(),
        'median_length': np.median(lengths),
        'min_length': lengths.min(),
        'max_length': lengths.max()
    })

tool_stats_df = pd.DataFrame(tool_stats)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Feature counts per tool (boxplot)
sns.boxplot(data=tool_stats_df, x='tool', y='num_features', ax=axes[0,0])
axes[0,0].set_ylabel('Number of Features Called', fontsize=12, fontweight='bold')
axes[0,0].set_xlabel('')
axes[0,0].set_title('A. Feature Count Distribution by Tool', fontsize=14, fontweight='bold', loc='left')
axes[0,0].set_yscale('log')
axes[0,0].grid(axis='y', alpha=0.3)

# Add median labels
medians = tool_stats_df.groupby('tool')['num_features'].median()
for i, tool in enumerate(medians.index):
    axes[0,0].text(i, medians[tool], f'{int(medians[tool]):,}', 
                   ha='center', va='bottom', fontweight='bold')

# 2. Mean feature length per tool
sns.boxplot(data=tool_stats_df, x='tool', y='mean_length', ax=axes[0,1])
axes[0,1].set_ylabel('Mean Feature Length (bp)', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('')
axes[0,1].set_title('B. Feature Length Distribution by Tool', fontsize=14, fontweight='bold', loc='left')
axes[0,1].grid(axis='y', alpha=0.3)

# 3. Feature counts per sample (stacked bar)
pivot = tool_stats_df.pivot_table(index='sample', columns='tool', values='num_features', fill_value=0)
pivot_top = pivot.sum(axis=1).nlargest(15).index
pivot.loc[pivot_top].plot(kind='barh', stacked=False, ax=axes[1,0], width=0.8)
axes[1,0].set_xlabel('Number of Features', fontsize=12, fontweight='bold')
axes[1,0].set_ylabel('')
axes[1,0].set_title('C. Features Called per Sample (Top 15)', fontsize=14, fontweight='bold', loc='left')
axes[1,0].set_xscale('log')
axes[1,0].legend(title='Tool', bbox_to_anchor=(1.05, 1), loc='upper left')
axes[1,0].grid(axis='x', alpha=0.3)

# 4. Summary table
summary = tool_stats_df.groupby('tool').agg({
    'num_features': ['median', 'mean'],
    'mean_length': 'mean',
    'median_length': 'mean'
}).round(0)
summary.columns = ['Median Features', 'Mean Features', 'Avg Mean Length', 'Avg Median Length']

axes[1,1].axis('off')
table = axes[1,1].table(cellText=summary.values.astype(int), 
                        rowLabels=summary.index, 
                        colLabels=summary.columns,
                        cellLoc='right', rowLoc='left',
                        loc='center', bbox=[0, 0, 1, 0.8])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2)
axes[1,1].set_title('D. Tool Summary Statistics', fontsize=14, fontweight='bold', loc='left', pad=20)

for (i, j), cell in table.get_celld().items():
    if i == 0:
        cell.set_text_props(weight='bold')
        cell.set_facecolor('#e0e0e0')
    if j == -1:
        cell.set_text_props(weight='bold')

plt.tight_layout()
plt.savefig('01_tool_calling_statistics.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("KEY INSIGHTS: Tool Calling Behavior")
print("="*60)
print(f"RiboTIE calls {medians['RiboTIE']/medians['ORFQuant']:.1f}x more features than ORFQuant")
print(f"Feature length differences suggest different thresholds/algorithms")

## 3. Agreement Overview

**Question**: Do tools agree? Which tool pairs have the most/least agreement?

In [ ]:
# Compute agreement patterns
agreement_data = []

for (sample, query_tool), df in consensus_data.items():
    match_cols = [c for c in df.columns if c.endswith('_both_matches')]
    other_tools = [c.replace('_both_matches', '') for c in match_cols]
    
    for other_tool in other_tools:
        start_matches = df[f'{other_tool}_start_matches']
        end_matches = df[f'{other_tool}_end_matches']
        both_matches = df[f'{other_tool}_both_matches']
        
        # Classify
        exact = (both_matches > 0).sum()
        start_only = ((start_matches > 0) & (both_matches == 0)).sum()
        end_only = ((end_matches > 0) & (both_matches == 0)).sum()
        neither = ((start_matches == 0) & (end_matches == 0)).sum()
        
        agreement_data.append({
            'sample': sample,
            'query_tool': query_tool,
            'other_tool': other_tool,
            'exact_match': exact,
            'start_only': start_only,
            'end_only': end_only,
            'no_match': neither,
            'total': len(df)
        })

agreement_df = pd.DataFrame(agreement_data)

# Calculate percentages
for col in ['exact_match', 'start_only', 'end_only', 'no_match']:
    agreement_df[f'{col}_pct'] = 100 * agreement_df[col] / agreement_df['total']

# Aggregate by tool pair
tool_pair_agg = agreement_df.groupby(['query_tool', 'other_tool']).agg({
    'exact_match': 'sum',
    'start_only': 'sum',
    'end_only': 'sum',
    'no_match': 'sum',
    'total': 'sum'
})

for col in ['exact_match', 'start_only', 'end_only', 'no_match']:
    tool_pair_agg[f'{col}_pct'] = 100 * tool_pair_agg[col] / tool_pair_agg['total']

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

# 1. Agreement type breakdown (stacked bars)
plot_data = tool_pair_agg[['exact_match_pct', 'start_only_pct', 'end_only_pct', 'no_match_pct']]
plot_data.columns = ['Exact Match', 'Start Only', 'End Only', 'No Match']

plot_data.plot(kind='barh', stacked=True, ax=axes[0,0],
               color=['#27ae60', '#f39c12', '#e67e22', '#95a5a6'], width=0.8)
axes[0,0].set_xlabel('Percentage of Features (%)', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('')
axes[0,0].set_title('A. Agreement Type by Tool Pair', fontsize=14, fontweight='bold', loc='left')
axes[0,0].legend(loc='lower right', frameon=True)
axes[0,0].set_xlim(0, 100)

# 2. Heatmap of exact match rates
pivot_exact = tool_pair_agg.reset_index().pivot(index='query_tool', columns='other_tool', values='exact_match_pct')
sns.heatmap(pivot_exact, annot=True, fmt='.2f', cmap='YlOrRd', ax=axes[0,1],
            cbar_kws={'label': 'Exact Match Rate (%)'})
axes[0,1].set_title('B. Exact Match Rates (Both Boundaries)', fontsize=14, fontweight='bold', loc='left')
axes[0,1].set_xlabel('')
axes[0,1].set_ylabel('')

# 3. Any agreement (start OR end OR both)
tool_pair_agg['any_agreement_pct'] = 100 - tool_pair_agg['no_match_pct']
pivot_any = tool_pair_agg.reset_index().pivot(index='query_tool', columns='other_tool', values='any_agreement_pct')
sns.heatmap(pivot_any, annot=True, fmt='.1f', cmap='RdYlGn', ax=axes[1,0],
            vmin=0, vmax=50, cbar_kws={'label': 'Any Agreement Rate (%)'})
axes[1,0].set_title('C. Any Agreement Rates (Start OR End OR Both)', fontsize=14, fontweight='bold', loc='left')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('')

# 4. Summary statistics
axes[1,1].axis('off')
summary_text = [
    "AGREEMENT SUMMARY (All Tool Pairs)",
    "=" * 45,
    "",
    f"Exact Match (both boundaries):  {agreement_df['exact_match_pct'].mean():.2f}%",
    f"Start Only:                     {agreement_df['start_only_pct'].mean():.2f}%",
    f"End Only:                       {agreement_df['end_only_pct'].mean():.2f}%",
    f"Any Agreement:                  {(100 - agreement_df['no_match_pct'].mean()):.2f}%",
    f"No Agreement:                   {agreement_df['no_match_pct'].mean():.2f}%",
    "",
    "KEY FINDING:",
    f"→ Tools agree on END more than START",
    f"  ({agreement_df['end_only_pct'].mean():.1f}% vs {agreement_df['start_only_pct'].mean():.1f}%)",
    "",
    "BEST TOOL PAIR:",
    f"→ iRibo ↔ RiboTIE: {pivot_exact.loc['iRibo', 'RiboTIE']:.2f}% exact",
]

axes[1,1].text(0.1, 0.9, '\n'.join(summary_text), 
               transform=axes[1,1].transAxes,
               fontsize=11, verticalalignment='top',
               fontfamily='monospace',
               bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))
axes[1,1].set_title('D. Summary Statistics', fontsize=14, fontweight='bold', loc='left', pad=20)

plt.tight_layout()
plt.savefig('02_agreement_overview.png', dpi=300, bbox_inches='tight')
plt.show()

## 4. Genomic Context Analysis

**Question**: Do tools agree more in certain genomic contexts (CDS, UTR, intergenic)?

In [ ]:
# Analyze agreement by genomic context
context_data = []

for (sample, query_tool), df in consensus_data.items():
    if 'cds_context' not in df.columns:
        continue
        
    match_cols = [c for c in df.columns if c.endswith('_both_matches')]
    other_tools = [c.replace('_both_matches', '') for c in match_cols]
    
    for other_tool in other_tools:
        for context in df['cds_context'].dropna().unique():
            ctx_df = df[df['cds_context'] == context]
            if len(ctx_df) == 0:
                continue
                
            start_matches = ctx_df[f'{other_tool}_start_matches']
            end_matches = ctx_df[f'{other_tool}_end_matches']
            both_matches = ctx_df[f'{other_tool}_both_matches']
            
            exact = (both_matches > 0).sum()
            any_agree = ((start_matches > 0) | (end_matches > 0)).sum()
            
            context_data.append({
                'sample': sample,
                'query_tool': query_tool,
                'other_tool': other_tool,
                'context': context,
                'exact_match': exact,
                'any_agreement': any_agree,
                'total': len(ctx_df)
            })

context_df = pd.DataFrame(context_data)
context_df['exact_match_pct'] = 100 * context_df['exact_match'] / context_df['total']
context_df['any_agreement_pct'] = 100 * context_df['any_agreement'] / context_df['total']

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Group by context and calculate means
context_summary = context_df.groupby('context').agg({
    'any_agreement_pct': 'mean',
    'total': 'sum'
}).reset_index()

# Sort by agreement rate
context_summary = context_summary.sort_values('any_agreement_pct', ascending=True)

# 1. Agreement by genomic context
bars = axes[0].barh(context_summary['context'], context_summary['any_agreement_pct'],
                     color=sns.color_palette("coolwarm", len(context_summary)))
axes[0].set_xlabel('Average Agreement Rate (%)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('Genomic Context', fontsize=12, fontweight='bold')
axes[0].set_title('A. Tool Agreement by Genomic Context', fontsize=14, fontweight='bold', loc='left')
axes[0].grid(axis='x', alpha=0.3)

# Add percentages on bars
for i, (idx, row) in enumerate(context_summary.iterrows()):
    axes[0].text(row['any_agreement_pct'] + 0.5, i, 
                 f"{row['any_agreement_pct']:.1f}%",
                 va='center', fontweight='bold')

# 2. Feature distribution by context
context_counts = context_df.groupby('context')['total'].sum().sort_values(ascending=False).head(10)
axes[1].bar(range(len(context_counts)), context_counts.values, 
            color=sns.color_palette("viridis", len(context_counts)))
axes[1].set_xticks(range(len(context_counts)))
axes[1].set_xticklabels(context_counts.index, rotation=45, ha='right')
axes[1].set_ylabel('Total Features', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Genomic Context', fontsize=12, fontweight='bold')
axes[1].set_title('B. Feature Distribution by Context (Top 10)', fontsize=14, fontweight='bold', loc='left')
axes[1].set_yscale('log')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig('03_genomic_context_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("KEY INSIGHTS: Genomic Context")
print("="*60)
top_context = context_summary.iloc[-1]
bottom_context = context_summary.iloc[0]
print(f"Best agreement in: {top_context['context']} ({top_context['any_agreement_pct']:.1f}%)")
print(f"Worst agreement in: {bottom_context['context']} ({bottom_context['any_agreement_pct']:.1f}%)")

## 5. Boundary-Specific Analysis

**Question**: Do tools systematically agree more on start or end boundaries? Is there evidence of overlapping signal?

In [ ]:
# Calculate boundary-specific agreement
agreement_df['start_agreement_pct'] = agreement_df['start_only_pct'] + agreement_df['exact_match_pct']
agreement_df['end_agreement_pct'] = agreement_df['end_only_pct'] + agreement_df['exact_match_pct']

# Plot
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Start vs End agreement scatter
scatter = axes[0,0].scatter(agreement_df['start_agreement_pct'], 
                            agreement_df['end_agreement_pct'],
                            c=agreement_df['exact_match_pct'], 
                            s=100, alpha=0.6, cmap='viridis',
                            edgecolors='black', linewidth=0.5)
axes[0,0].plot([0, 100], [0, 100], 'k--', alpha=0.3, label='Equal agreement')
axes[0,0].set_xlabel('Start Agreement Rate (%)', fontsize=12, fontweight='bold')
axes[0,0].set_ylabel('End Agreement Rate (%)', fontsize=12, fontweight='bold')
axes[0,0].set_title('A. Start vs End Boundary Agreement', fontsize=14, fontweight='bold', loc='left')
axes[0,0].legend()
axes[0,0].grid(alpha=0.3)
cbar = plt.colorbar(scatter, ax=axes[0,0])
cbar.set_label('Exact Match %', rotation=270, labelpad=20)

# Add annotation for points above/below diagonal
above_diag = (agreement_df['end_agreement_pct'] > agreement_df['start_agreement_pct']).sum()
below_diag = (agreement_df['end_agreement_pct'] < agreement_df['start_agreement_pct']).sum()
axes[0,0].text(0.05, 0.95, f"Above diagonal (End > Start): {above_diag}\nBelow diagonal (Start > End): {below_diag}",
               transform=axes[0,0].transAxes, fontsize=10,
               verticalalignment='top',
               bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))

# 2. Tool-specific boundary preferences
tool_boundary = agreement_df.groupby('query_tool')[['start_agreement_pct', 'end_agreement_pct']].mean()
tool_boundary.plot(kind='bar', ax=axes[0,1], width=0.8, color=['#3498db', '#e74c3c'])
axes[0,1].set_ylabel('Agreement Rate (%)', fontsize=12, fontweight='bold')
axes[0,1].set_xlabel('Tool', fontsize=12, fontweight='bold')
axes[0,1].set_title('B. Average Boundary Agreement by Tool', fontsize=14, fontweight='bold', loc='left')
axes[0,1].legend(['Start Agreement', 'End Agreement'])
axes[0,1].set_xticklabels(axes[0,1].get_xticklabels(), rotation=45, ha='right')
axes[0,1].grid(axis='y', alpha=0.3)

# 3. Boundary bias (end - start)
agreement_df['boundary_bias'] = agreement_df['end_agreement_pct'] - agreement_df['start_agreement_pct']
tool_pair_bias = agreement_df.groupby(['query_tool', 'other_tool'])['boundary_bias'].mean().reset_index()
pivot_bias = tool_pair_bias.pivot(index='query_tool', columns='other_tool', values='boundary_bias')

sns.heatmap(pivot_bias, annot=True, fmt='.1f', cmap='RdBu_r', center=0,
            ax=axes[1,0], cbar_kws={'label': 'Boundary Bias (End - Start)'})
axes[1,0].set_title('C. Boundary Bias Heatmap\n(Positive = End-biased, Negative = Start-biased)', 
                    fontsize=14, fontweight='bold', loc='left')
axes[1,0].set_xlabel('')
axes[1,0].set_ylabel('')

# 4. Disagreement type distribution
disagree_types = agreement_df[['start_only_pct', 'end_only_pct', 'no_match_pct']].mean()
disagree_types.index = ['Start Only\n(disagree on end)', 'End Only\n(disagree on start)', 'No Match']

colors = ['#f39c12', '#e67e22', '#95a5a6']
wedges, texts, autotexts = axes[1,1].pie(disagree_types.values, labels=disagree_types.index,
                                          autopct='%1.1f%%', colors=colors, startangle=90)
axes[1,1].set_title('D. Types of Disagreement\n(When Tools Disagree)', 
                    fontsize=14, fontweight='bold')

for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

plt.tight_layout()
plt.savefig('04_boundary_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("KEY INSIGHTS: Boundary Agreement")
print("="*60)
print(f"Average Start Agreement: {agreement_df['start_agreement_pct'].mean():.2f}%")
print(f"Average End Agreement:   {agreement_df['end_agreement_pct'].mean():.2f}%")
print(f"\n→ Tools agree {agreement_df['end_agreement_pct'].mean() / agreement_df['start_agreement_pct'].mean():.1f}x more on ENDS than STARTS")
print(f"→ This suggests stop codon signal is clearer than start codon signal")

## 6. Tool Pair Comparison Matrix

**Question**: Which specific tool pairs agree most? Which disagree most?

In [ ]:
# Create comprehensive tool pair comparison
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()

# Calculate metrics for each tool pair
tool_pairs = agreement_df.groupby(['query_tool', 'other_tool']).agg({
    'exact_match_pct': 'mean',
    'start_agreement_pct': 'mean',
    'end_agreement_pct': 'mean',
    'start_only_pct': 'mean',
    'end_only_pct': 'mean',
    'no_match_pct': 'mean'
}).reset_index()

metrics = [
    ('exact_match_pct', 'Exact Match Rate', 'Greens'),
    ('start_agreement_pct', 'Start Agreement', 'Blues'),
    ('end_agreement_pct', 'End Agreement', 'Reds'),
    ('start_only_pct', 'Start Only (End Disagree)', 'Oranges'),
    ('end_only_pct', 'End Only (Start Disagree)', 'Purples'),
    ('no_match_pct', 'No Agreement', 'Greys')
]

for idx, (metric, title, cmap) in enumerate(metrics):
    pivot = tool_pairs.pivot(index='query_tool', columns='other_tool', values=metric)
    sns.heatmap(pivot, annot=True, fmt='.2f', cmap=cmap, ax=axes[idx],
                cbar_kws={'label': 'Rate (%)'})
    axes[idx].set_title(title, fontsize=12, fontweight='bold')
    axes[idx].set_xlabel('')
    axes[idx].set_ylabel('')

plt.tight_layout()
plt.savefig('05_tool_pair_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

# Print best/worst pairs
print("\n" + "="*60)
print("BEST & WORST TOOL PAIRS")
print("="*60)
best_exact = tool_pairs.nlargest(3, 'exact_match_pct')
print("\nBest Exact Agreement:")
for _, row in best_exact.iterrows():
    print(f"  {row['query_tool']:10s} ↔ {row['other_tool']:10s}: {row['exact_match_pct']:.2f}%")

worst_agreement = tool_pairs.nlargest(3, 'no_match_pct')
print("\nWorst Agreement (Most No-Match):")
for _, row in worst_agreement.iterrows():
    print(f"  {row['query_tool']:10s} ↔ {row['other_tool']:10s}: {row['no_match_pct']:.1f}% no match")

## 7. Export Results

In [ ]:
# Export summary tables
output_dir = Path('/homes/jackt/consensus_analysis_output')
output_dir.mkdir(exist_ok=True)

tool_stats_df.to_csv(output_dir / 'tool_calling_statistics.tsv', sep='\t', index=False)
agreement_df.to_csv(output_dir / 'tool_agreement_data.tsv', sep='\t', index=False)
context_df.to_csv(output_dir / 'context_agreement_data.tsv', sep='\t', index=False)
tool_pairs.to_csv(output_dir / 'tool_pair_summary.tsv', sep='\t', index=False)

print(f"Results exported to: {output_dir}")
print("\nGenerated files:")
print("  - tool_calling_statistics.tsv")
print("  - tool_agreement_data.tsv")
print("  - context_agreement_data.tsv")
print("  - tool_pair_summary.tsv")